# Data Exploration

Inspect curated output at each pipeline stage. Answers the question:
**did my curation actually work?**

- Doc length distributions
- Language score distributions
- Retention rates per stage
- Sample documents before/after filtering
- Token count estimates

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

STAGES_DIR = Path("/data/curated/stages")
STAGES = ["extract", "language_filter", "heuristic_filter",
          "exact_dedup", "fuzzy_dedup", "pii"]

print("Stages found:")
for stage in STAGES:
    d = STAGES_DIR / stage
    files = list(d.glob("*.jsonl")) if d.exists() else []
    complete = (d / ".complete").exists() if d.exists() else False
    print(f"  {stage:<20} {'✓ complete' if complete else '○ incomplete'} — {len(files)} JSONL file(s)")

## Retention Funnel

In [ ]:
def count_docs(stage: str) -> int:
    stage_dir = STAGES_DIR / stage
    if not stage_dir.exists():
        return 0
    total = 0
    for f in stage_dir.glob("*.jsonl"):
        with open(f) as fh:
            total += sum(1 for line in fh if line.strip())
    return total

counts = {stage: count_docs(stage) for stage in STAGES}
print("Stage document counts:")
print(f"  {'Stage':<25} {'Docs':>10} {'Retention':>12}")
print("  " + "-" * 50)

first = None
for stage, count in counts.items():
    if first is None and count > 0:
        first = count
    pct = f"{count/first*100:.1f}%" if first else "—"
    print(f"  {stage:<25} {count:>10,} {pct:>12}")

In [ ]:
# Retention funnel chart
valid = {k: v for k, v in counts.items() if v > 0}
if len(valid) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    stages = list(valid.keys())
    values = list(valid.values())
    bars = ax.bar(stages, values, color="#5DCAA5", edgecolor="#1D9E75", linewidth=0.5)
    ax.set_ylabel("Document count")
    ax.set_title("Retention funnel — documents surviving each stage")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f"{val:,}", ha="center", va="bottom", fontsize=9)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig("/results/eval/retention_funnel.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough stage data to plot — run curation first.")

## Document Length Distribution

In [ ]:
def sample_docs(stage: str, n: int = 2000) -> list[dict]:
    stage_dir = STAGES_DIR / stage
    if not stage_dir.exists():
        return []
    docs = []
    for f in stage_dir.glob("*.jsonl"):
        with open(f) as fh:
            for line in fh:
                line = line.strip()
                if line:
                    docs.append(json.loads(line))
    random.shuffle(docs)
    return docs[:n]

# Compare word counts before and after heuristic filtering
extract_docs  = sample_docs("extract")
filtered_docs = sample_docs("heuristic_filter")

if extract_docs and filtered_docs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, docs, label, color in [
        (axes[0], extract_docs,  "After extract",           "#85B7EB"),
        (axes[1], filtered_docs, "After heuristic filter",  "#5DCAA5"),
    ]:
        word_counts = [len(d.get("text", "").split()) for d in docs]
        ax.hist(word_counts, bins=50, color=color, edgecolor="white", linewidth=0.3)
        ax.set_xlabel("Word count")
        ax.set_ylabel("Documents")
        ax.set_title(f"{label} (n={len(docs):,})")
        ax.axvline(np.median(word_counts), color="#2C2C2A", linestyle="--",
                   linewidth=1, label=f"Median: {int(np.median(word_counts))}")
        ax.legend(fontsize=9)
    plt.suptitle("Document word count distribution", fontsize=12)
    plt.tight_layout()
    plt.savefig("/results/eval/word_count_dist.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Run curation to generate data for this analysis.")

## Language Score Distribution

In [ ]:
lang_docs = sample_docs("language_filter", n=5000)

if lang_docs and "language_score" in lang_docs[0]:
    scores = [d["language_score"] for d in lang_docs]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(scores, bins=40, color="#AFA9EC", edgecolor="white", linewidth=0.3)
    ax.axvline(0.65, color="#D85A30", linestyle="--", linewidth=1.5, label="Threshold (0.65)")
    ax.set_xlabel("fastText language confidence score")
    ax.set_ylabel("Documents")
    ax.set_title("English language score distribution (after language filter)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Language scores not found — check language_filter stage output.")

## Sample Documents

In [ ]:
# Print random samples from the final PII stage
pii_docs = sample_docs("pii", n=100)

N_SHOW = 3
for i, doc in enumerate(random.sample(pii_docs, min(N_SHOW, len(pii_docs)))):
    print(f"\n{'='*60}")
    print(f"URL:    {doc.get('url', 'N/A')}")
    print(f"Words:  {doc.get('word_count', '?')} | Chars: {doc.get('char_count', '?')}")
    if doc.get('language_score'):
        print(f"Lang score: {doc['language_score']}")
    print(f"\nText (first 500 chars):")
    print(doc.get("text", "")[:500])

## Token Count Estimate

In [ ]:
# Estimate total tokens from mmap files
import struct
import numpy as np

tokenized_dir = Path("/data/curated/tokenized")
idx_files = list(tokenized_dir.glob("*.idx"))

if idx_files:
    total_tokens = 0
    total_docs = 0
    for idx_file in idx_files:
        with open(idx_file, "rb") as f:
            f.read(9 + 8 + 1)  # skip header
            doc_count = struct.unpack("<Q", f.read(8))[0]
            sizes = np.frombuffer(f.read(doc_count * 4), dtype=np.int32)
        total_tokens += sizes.sum()
        total_docs   += doc_count
        print(f"{idx_file.name}: {doc_count:,} docs, {sizes.sum()/1e6:.1f}M tokens")
    print(f"\nTotal: {total_docs:,} documents — {total_tokens/1e9:.3f}B tokens")
    print(f"Target for 125M (Chinchilla): ~2.5B tokens")
    print(f"Coverage: {total_tokens/2.5e9*100:.1f}% of Chinchilla-optimal")
else:
    print("No tokenized files found. Run 'make tokenizer' first.")